# 01 数值微分问题概览

数值微分要解决的问题是：在只掌握有限函数值或有限离散数据时，如何近似

$$
f'(x),\qquad f''(x).
$$

这与数值积分不同。积分会把局部误差平均或累积，微分则常常把函数值误差除以步长，因此更容易放大噪声和舍入误差。本章的核心不是背公式，而是理解公式从哪里来、误差如何变化、什么时候应该谨慎使用。


## 函数型微分和离散数据微分

本章把输入分成两类。

**函数型微分**：可以按需计算 $f(x)$。这时可以主动选择步长 $h$，并用步长减半实验估计误差阶。

**离散数据微分**：只给出

$$
(x_i,y_i),\qquad y_i\approx f(x_i).
$$

这时不能随意缩小步长，只能使用已有节点。若 $y_i$ 含噪声，差分会把噪声放大，尤其在二阶导数中更明显。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import central_difference, forward_difference, observed_order


## 一个最小例子：前向差分和中心差分

前向差分来自极限定义：

$$
D_+f(x)=\frac{f(x+h)-f(x)}{h}.
$$

中心差分使用两侧点：

$$
D_0f(x)=\frac{f(x+h)-f(x-h)}{2h}.
$$

Taylor 展开显示前向差分通常是一阶公式，中心差分通常是二阶公式。下面先用一个教学式实现观察这个差别。


In [ ]:
def teaching_forward_difference(f, x, h):
    return (f(x + h) - f(x)) / h


def teaching_central_difference(f, x, h):
    return (f(x + h) - f(x - h)) / (2 * h)

x0 = 0.7
exact = math.cos(x0)
for h in [0.2, 0.1, 0.05, 0.025]:
    e_forward = abs(teaching_forward_difference(math.sin, x0, h) - exact)
    e_central = abs(teaching_central_difference(math.sin, x0, h) - exact)
    print(f"h={h:7.4f}  前向误差={e_forward:.3e}  中心误差={e_central:.3e}")


## 截断误差与舍入误差的竞争

步长 $h$ 变小时，截断误差通常下降；但差分公式包含相减和除以 $h$，过小的 $h$ 会让舍入误差显现。一个粗略模型是

$$
E(h)\approx C_1h^p+C_2\frac{\epsilon}{h},
$$

其中 $p$ 是差分公式的截断误差阶，$\epsilon$ 表示函数值计算中的浮点误差量级。这个模型解释了数值微分中常见的 U 形误差曲线。


In [ ]:
x0 = 1.0
exact = math.exp(x0)
h_values = 10.0 ** (-np.arange(1, 17, dtype=float))
errors = np.array([abs(teaching_central_difference(math.exp, x0, h) - exact) for h in h_values])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(h_values, errors, marker="o")
ax.invert_xaxis()
ax.set_xlabel("步长 h")
ax.set_ylabel("绝对误差")
ax.set_title("中心差分误差的 U 形变化")
ax.grid(True, which="both", alpha=0.3)
plt.show()


## 教学实现与正式实现对照

Notebook 中保留教学式实现，目的是让公式和代码一一对应。`src/py_sc/differentiation.py` 中的正式实现会增加输入检查、数组支持和可复用接口。下面比较两者在同一问题上的结果。


In [ ]:
h = 0.05
teaching_value = teaching_central_difference(math.sin, x0, h)
script_value = float(central_difference(np.sin, x0, h))
print("教学实现：", teaching_value)
print("正式实现：", script_value)
print("差异：", abs(teaching_value - script_value))


## 本章阅读顺序

1. 先阅读有限差分公式，理解 Taylor 展开如何给出系数和误差阶。
2. 再阅读离散数据微分，区分可主动选步长和只能使用已有节点的情形。
3. 然后阅读样条微分，理解“先构造平滑函数，再解析求导”的思路。
4. Richardson 外推和隐式格式用于展示如何提高精度，但也要同时关注稳定性。
5. 最后通过统一实验观察收敛阶、U 形误差曲线和噪声放大。

本章的很多思想会直接进入后续常微分方程和偏微分方程章节。
